In [1]:
from pymarc import Record, MARCReader, Subfield, Reader, Field
from collections import defaultdict
from pymarc import Record, MARCReader, Subfield, map_xml

In [33]:
authors = {}
translators = set()
trl_code = {}
with open(f'data/marc_bibliografie_prekladu_a_d_opraveno.mrc', 'rb') as data:
    reader = MARCReader(data, to_unicode=True, force_utf8=True, utf8_handling="strict")
    for i, record in enumerate(reader): 
                    for field in record.get_fields('100'):
                        subfields = field.subfields_as_dict()
                        if 'a' in subfields.keys():
                            aut = subfields['a'][0]
                            if aut not in authors.keys():
                                authors[aut] = None
                        if '7' in subfields.keys():
                            code = subfields['7'][0] 
                            if authors[aut] is not None:
                                if code not in authors[aut]: authors[aut].add(code) 
                            else: authors[aut] = set([code])     

                    for field in record.get_fields('700'):
                        subfields = field.subfields_as_dict()
                        if 'a' in subfields.keys():
                            trl = subfields['a'][0].strip(', ')
                            translators.add(trl)
                        if '7' in subfields.keys():
                            trl_code[subfields['7'][0]] = trl   
for key, value in authors.items():
    if value is not None and len(value) > 1: print(f'{key}: {value}') 
for key, value in authors.items():
    if value is None: print(key)                                
                 

Ciprová, Inka
Bradáč, František
Červinka, Vincenc
Dvořáček, Jaroslav
Marek, Jiří
Zima, František
Gromek, Jiří
Kratochvil, Jiří
Anjelakis, Andreas
Baletas, Kostas
Amaniti, N.
Ekrtová, Eva
Garaudy, Roger
Klaus, Václav
Úřad vlády ČR. Sekretariát Rady vlády pro národnostní menšiny
Kana, Sofia
Solženicyn, Aleksandr
Dvořák, Jiří
Chlup, Marek
Dordanas, Stratos N.
Lehar, František
Jyftakis, E. Sotiris
Štubňa, Antonín
Hlaváček, B.
Lec, Stanislav Jerzy
Kubelka, B.
Kocourek, B. K.
Kubálek, Jan
Jedlička, Josef
Giudici, Giovanni
Cosentino, Annalisa
Parente, Antonio


In [95]:
correct_dict = {'Fučík, Julius,': 'jk01032106', 
                'Škvorecký, Josef,': 'jk01130413', 
                'Kolář, Jiří,' : 'jk01061244',
                'Hašek, Jaroslav,': 'jk01040097',
                'Legátová, Květa,': 'jk01100201',
                'Hájek, Jiří,': 'jk01033052',
                'Wolker, Jiří,': 'jk01151789',
                'Holan, Vladimír,':'jk01041653',
                'Hruška, Petr,': 'jn19990209272',
                'Orten, Jiří,': 'jk01091274'}
for key, value in authors.items():
    if value is not None and len(value) == 1: correct_dict[key] = list(value)[0]
corrected_years = {}
correct_dict = {aut.strip(', '):value for aut,value in correct_dict.items()}

In [96]:
OUT = 'data/marc_bibliografie_prekladu_a_d_codes.mrc'
with open(OUT , 'wb') as writer:
        with open(f'data/marc_bibliografie_prekladu_a_d.mrc', 'rb') as data:
            reader = MARCReader(data, to_unicode=True, force_utf8=True, utf8_handling="strict")
            for i, record in enumerate(reader): 
                for field in record.get_fields('100'):
                        subfields = field.subfields_as_dict()
                        if 'a' in subfields.keys():
                              aut = subfields['a'][0].strip(', ')
                              if aut in correct_dict.keys():
                                    if '7' in subfields.keys() and subfields['7'][0].strip() == correct_dict[aut]:
                                        if 'd' in subfields.keys(): corrected_years[aut] =  subfields['d'][0]

In [117]:
def change_595(record, aut):
    for field in record.get_fields('595'):
        subfields = field.subfields_as_dict()
        if '7' in subfields.keys() :
            if aut in corrected_years.keys():
                if 'd' in subfields.keys() : record['595']['d'] = corrected_years[aut]
                else: field.add_subfield(code = 'd', value = corrected_years[aut], pos=1) 
                record['595']['7'] = correct_dict[aut]
            else: record['595']['7'] = correct_dict[aut]
        else:
            if aut in corrected_years.keys():
                field.add_subfield(code = 'd', value = corrected_years[aut], pos=1) 
                field.add_subfield(code = '7', value = correct_dict[aut], pos=2)
            else: field.add_subfield(code = '7', value = correct_dict[aut], pos=1) 
    return record         


def change_author_code(record):
    for field in record.get_fields('100'):
        subfields = field.subfields_as_dict()
        if 'a' in subfields.keys():
            aut = subfields['a'][0].strip(', ')
            if aut in correct_dict.keys():
                if '7' in subfields.keys() and subfields['7'][0].strip() != correct_dict[aut]:
                    record = change_595(record, aut)
                    record.remove_field(field)
                    if aut in corrected_years.keys():record.add_ordered_field(Field(tag = '100', indicators=['1', ' '], subfields=[Subfield(code = 'a', value = aut ),
                                                                                                    Subfield(code = 'd', value = corrected_years[aut]),
                                                                                                    Subfield(code = '7', value = correct_dict[aut]),
                                                                                                    Subfield(code = '4', value = 'aut')]))
                    else: record.add_ordered_field(Field(tag = '100', indicators=['1', ' '], subfields=[Subfield(code = 'a', value = aut ),
                                                                                                    Subfield(code = '7', value = correct_dict[aut]),
                                                                                                    Subfield(code = '4', value = 'aut')]))
                elif '7' not in subfields.keys():
                    record = change_595(record, aut)
                    if aut in corrected_years.keys():
                        field.add_subfield(code = 'd', value = corrected_years[aut], pos=1) 
                        field.add_subfield(code = '7', value = correct_dict[aut], pos=2)
                    else: field.add_subfield(code = '7', value = correct_dict[aut], pos=1)  
    return record                


In [118]:
OUT = 'data/marc_bibliografie_prekladu_a_d_codes.mrc'
with open(OUT , 'wb') as writer:
        with open(f'data/marc_bibliografie_prekladu_a_d.mrc', 'rb') as data:
            reader = MARCReader(data, to_unicode=True, force_utf8=True, utf8_handling="strict")
            for i, record in enumerate(reader): 
                
                ### 007
                for field in record.get_fields('007'):
                    record.remove_field(field)

                for field in record.get_fields('041'):
                    subfields = field.subfields_as_dict()
                    if 'h' not in subfields.keys():
                        field.add_subfield(code = 'h', value = 'cze', pos = 1)
                         

                ### Change Authors code
                print(record)
                record = change_author_code(record)                  
                print(record)
                
                ### 500 Velkým písmenem a konec tečka
                for field in record.get_fields('500'):
                    ind1 = field.indicator1
                    ind2 = field.indicator2
                    subfield = field.subfields_as_dict()
                    record.remove_field(field)
                    s = subfield['a'][0].capitalize()
                    s = s.strip('.') + '.'
                    record.add_ordered_field(Field(tag = '500', indicators=[ind1, ind2], subfields=[Subfield(code = 'a', value = s)]))
                
                ### Originál neznámý dát do poznámky
                for field in record.get_fields('595'):
                    subfield = field.subfields_as_dict()
                    if 't' in subfield:
                        if subfield['t'][0].lower() =='originál neznámý.' or subfield['t'][0].lower() == 'originál neexistuje.' or subfield['t'][0].lower() == 'originál nenalezen.' :
                            s = subfield['t'][0].capitalize()
                            s = s.strip('.') + '.'
                            record.remove_field(field)
                            record.add_ordered_field(Field(tag = '500', indicators=[' ', ' '], subfields=[Subfield(code = 'a', value = s)]))  

                ### Change TRL                                                                                                         
                record['964']['a'] = 'TRL2'                                
                writer.write(record.as_marc())                        

=LDR  01475nam-a--003854i-4500
=001  trl0012787
=003  CZ\PrUCL
=005  20010918000000.0
=008  010918s1977----fi-----j----------f-fin-d
=020  \\$a951-0-08348-8
=035  \\$a(SKC)bknzdr00204
=040  \\$aABB060$bcze$erda
=041  1\$afin$hcze
=100  1\$aBořkovcová, Hana,$d1927-2009$7jk01012732$4aut
=240  10$aMy tři cvoci.$lFinsky
=245  10$aMe kolme pöhköä / $c[Autorka:] Hanna Knapp ; Suomentanut Kirsti Siraste
=264  \\$aPorvoo :$bWerner Söderström, Osakeyhtiö,$c1977$f(WSOY)
=300  \\$a169, [1] s. ;$c8°
=490  1\$aNuorten Toive-kirjasto ;$vNo 245
=500  \\$aIl. předsádky
=500  \\$aTšekinkielinen alkuteos: My tři cvoci
=500  \\$aAutorka publikuje pod čes. jménem Hana Bořkovcová
=561  \\$aHana Hlinovská: Bibliografie překladů české literatury do finštiny. Bakalářská práce. FF MU, Brno 2024.
=595  12$aBořkovcová, Hana,$d1927-2009$7jk01012732$tMy tři cvoci$1ubcjk013280923
=700  1\$aBořkovcová, Hana$4ill
=700  1\$aSiraste, Kirsti$4trl
=830  \0$aNuorten Toive-kirjasto
=900  \\$

### NAPOJIT AUT

In [51]:
official_names_aut = {}
birth_year_aut = {}
non_official_names_aut = defaultdict(list)
additonal_info_aut = defaultdict(list)

def do_it(record):
    global official_names_aut, non_official_names_aut, birth_year_aut, additonal_info_aut
    if not record is None:
                birth = 0
                birth_dead = 0
                for field in  record.get_fields('100') : 
                    insert = [None] * 4
                    subfields = field.subfields_as_dict()
                    if 'd' in subfields.keys():
                        for birth in field.get_subfields('d'):
                            birth_dead = birth
                            birth = 1700 if any([x.isalpha() for x in birth ]) else int(str(birth[:4]).replace('?', '0').replace('-', ''))
                    else:     
                        birth = 0 
                    for code in field.get_subfields('7'):
                        official_names_aut[code] = field['a'].strip(',').strip()
                        birth_year_aut[code] = birth_dead
                        insert[0] = str(field['a'].strip(',').strip())
                        insert[1] = birth_dead
                        alternative_names = []
                        for field in  record.get_fields('400') :
                            non_official_names_aut[code].append(field['a'])
                            alternative_names.append(str(field['a']))
                        insert[2] = alternative_names
                        for field in record.get_fields('678'): 
                            insert[3] = str(field['a'])
                        additonal_info_aut[code] = insert  
map_xml(do_it, 'data/aut_ja.xml.gz.xml')                          

### TODO: trl2 + 500 velkým písmenem a tečka + originál neznámý smazat a dát do poznámky + 035?? 

In [27]:
reverse_official_names = defaultdict(list)
for key, value in official_names_aut.items():
    reverse_official_names[value].append(key)

In [44]:
n = 0
true_trans_code = {'Kohout, Pavel': 'jk01061137',
              'Dean, Adrian': 'ola2018991701',
              'Lada, Josef': 'jk01071364',
              'Holub, Miroslav': 'jk01041818',
              'Řezáč, Jan':'jk01110101'}
for trans in translators: 
    if trans in reverse_official_names.keys():
        if len(reverse_official_names[trans]) == 1:
            true_trans_code[trans] = reverse_official_names[trans][0]
        else: print(f'{trans}: {reverse_official_names[trans]}')


#### 

In [45]:
i = 0
for key, value in authors.items():
    if value is None: 
        continue
    key = key.strip(', ')
    if key in translators:
        trl_code[list(value)[0]] = key
        i+= 1
        print(key)
print(f'Found names: {i}\n')

#### Find translators that do not have code
for _, trl_name in trl_code.items():
    if isinstance(trl_name, list):
        if trl_name[0] in translators:
            translators.remove(trl_name[0])
    else: 
        if trl_name in translators: translators.remove(trl_name)    
 
for _, trl_name in trl_code.items():
    if isinstance(trl_name, list):
        trl_name = trl_name[0].strip(', ')
        if trl_name in translators:
            print(trl_name)
            translators.remove(trl_name)
    else: 
        trl_name = trl_name.strip(', ')
        if trl_name in translators: 
            translators.remove(trl_name)   
            print(trl_name) 
translators = [v.strip(', ') for v in list(translators)]
for key, value in authors.items():
    if value is None: 
        continue
    key = key.strip(', ')
    if key in translators:
        trl_code[list(value)[0]] = key
        i+= 1
        print(key)    
reverse_trl_code = {}
for key, value in trl_code.items():
    reverse_trl_code[value] = key  
for key, value in true_trans_code.items():
    reverse_trl_code[key] = value              

Found names: 0



In [46]:
for key, _ in reverse_trl_code.items(): print(key)

Siraste, Kirsti
Balk, Eero
Manner, Eeva-Liisa
Hegar, Milan
Burian, Zdeněk
Voipio, Paavo
Šťovíčková, Milada
Bessonoff, Leo
Teissig, Karel
Troup, Miloslav
Senius, Kari
Hvížďala, Karel
Pallasmaa, Juhani
Carpelan, Bo
Dean, Adrian
Bermel, Neil
Silvanto, Reino
Krohn, Leena
Miler, Zdeněk
Uusitalo, Maire
Blomstedt, Jan
Klemke, Werner
Lada, Josef
Sinervo, Elvi
Macourek, Miloš
Vyskočil, Ivan
Krumbachová, Ester
Mikulka, Alois
Sekyrková, Hana
Hejná, Olga
Borská, Ilona
Vostrá, Alena
Franková, Hermína
Čtvrtek, Václav
Aškenazy, Ludvík
Štuka, Ivo
Anhava, Helena
Manninen, Kerttu
Janáček, Leoš
Sabina, Karel
Vladislav Jan
Meyer-Rey, Ingeborg
Tapola, Jussi
Kvapil, Jaroslav
Pavlík, Milan
Hofman, Eduard
Smetana, Bedřich
Kejř, Jiří
Hrabal, Bohumil
Kundera, Milan
Olbracht, Ivan
Tsizek, Karolos
Forman, Miloš
Papadopulos, Lysimachos
Kohout, Pavel
Nollas, Dimitris
Pupti, Eleni
Tsivos, Kostas
Čapek, Karel
Pekárek, Karel
Garaudy, Roger
Goodman, John C.
Miller, Arthur
Riese, Hans-

In [43]:
for t in list(translators): print(t)

Somerova, Vera
Všetečka, Jiří
Kudel, Silja
Trevisan, Alessandra
Wallenius, Toivo
Basaran, Mehmet
Lada, josef
Kulmala, Elli
Evanjelatos, Spyros A.
Tsakalidu, Anna
Räty, Susanna
Fortunato, Lara
Brignole, Francesco
Ahlström-Taavitsainen, Camilla
Darviras, Marios
Meyer, J.
Nikolski, S.
Korzankova-Sarandopulu, Jitka
Moschos, Nikos
Zachovalová, Lieko
Huuhka, Matti
Kurenniemi, Marjatta
Čech, Ital
Anjelaki, Despina
De Nardis, Luisa
Miler, Kateřina
Motyčka, Jiří
Dvořák, Václav
Kurtovik, Dimosthenis
Lehto, Pentti
Želibská, Mária
Anemojannis, G.
Pullinen, Erkki
Luumi, Johanna
Kuvelis, Fotis
Kolibal, Stanislav
Savvidis, G. P.
Sklenář, Zdeněk
Divani, L.
Skambeta, A. A.
Schweiss, Petra
Lalos, Andonis
Kukiu, Chrysula
Fertaki, Annika
Puntari, Jukka H.
Juvonen, Riikka
Dadone, Viktorka
Polák, Ondřej
Dimitriadu, Androniki
Asonitis, A. M.
Kábrt, Josef
Roinila, Pirkko
Anjelakis, Andreas
Ferrario, Andrea
Chandra, G.
Chatziprodromidis, Leonidas
Klondza-Jaklova, Vera
Lužík, R.
Nianio

In [ ]:
OUT = 'data/marc_bibliografie_prekladu_a_d_add_trl.mrc'
with open(OUT , 'wb') as writer:
        with open(f'data/marc_bibliografie_prekladu_a_d_opraveno.mrc', 'rb') as data:
            reader = MARCReader(data, to_unicode=True, force_utf8=True, utf8_handling="strict")
            for i, record in enumerate(reader): 
                for field in record.get_fields('700'):
                        subfields = field.subfields_as_dict()
                        if 'a' in subfields.keys():
                              aut = subfields['a'][0].strip(', ')
                              if aut in true_trans_code.keys():
                                sub = [Subfield(code = 'a', value = aut)]
                                if '6' in subfields.keys(): 
                                    sub2 = sub
                                    sub = [Subfield(code = '6', value = subfields['6'][0])]
                                    sub.append(sub2[0])
                                if true_trans_code[aut] in birth_year_aut.keys(): sub.append(Subfield(code = 'd', value = birth_year_aut[true_trans_code[aut]] if '-' in str(birth_year_aut[true_trans_code[aut]]) else str(birth_year_aut[true_trans_code[aut]]) + '-' ))
                                sub.append(Subfield(code = '7', value = true_trans_code[aut] ))
                                if '4' in subfields.keys(): sub.append(Subfield(code = '4', value = subfields['4'][0]))
                                record.add_ordered_field(Field(tag = '700', indicators=['1', ' '], subfields=sub))      
                                record.remove_field(field)
                writer.write(record.as_marc())     


In [54]:
OUT = 'data/marc_bibliografie_prekladu_a_d_add_trl.mrc'
with open(OUT , 'wb') as writer:
        with open(f'data/marc_bibliografie_prekladu_a_d_opraveno.mrc', 'rb') as data:
            reader = MARCReader(data, to_unicode=True, force_utf8=True, utf8_handling="strict")
            for i, record in enumerate(reader): 
                for field in record.get_fields('700'):
                        subfields = field.subfields_as_dict()
                        if 'a' in subfields.keys():
                              aut = subfields['a'][0].strip(', ')
                              if aut in reverse_trl_code.keys():
                                sub = [Subfield(code = 'a', value = aut+', ')]
                                if '6' in subfields.keys(): 
                                    sub2 = sub
                                    sub = [Subfield(code = '6', value = subfields['6'][0])]
                                    sub.append(sub2[0])
                                if reverse_trl_code[aut] in birth_year_aut.keys(): sub.append(Subfield(code = 'd', value = birth_year_aut[reverse_trl_code[aut]] if '-' in str(birth_year_aut[reverse_trl_code[aut]]) else str(birth_year_aut[reverse_trl_code[aut]]) + '-' ))
                                sub.append(Subfield(code = '7', value = reverse_trl_code[aut] ))
                                if '4' in subfields.keys(): sub.append(Subfield(code = '4', value = subfields['4'][0]))
                                record.add_ordered_field(Field(tag = '700', indicators=['1', ' '], subfields=sub))      
                                record.remove_field(field)
                writer.write(record.as_marc())     

In [52]:
print(birth_year_aut['jk01110101'])

1921-2009
